# `param_shift` walkthrough

Build a small real torchsynth dataset, add the `shift` sensitivity column, and browse the
result in SmooSense.

`param_shift` assigns every row exactly one parameter of the synth's param spec, redraws
that parameter from its own distribution, re-renders the patch, and scores the perturbed
audio against the row's stored audio. The seven facets land in one nested `shift` struct:

| Subfield | Meaning |
| --- | --- |
| `shift.param` | Name of the one parameter shifted on this row |
| `shift.amount` | Size of the shift in encoded space (L2 over the parameter's span) |
| `shift.audio` | Audio rendered from the shifted patch |
| `shift.rms` | RMS-envelope cosine **similarity** — 1.0 means unchanged |
| `shift.sot` | Sliced-optimal-transport **distance** |
| `shift.wmfcc` | DTW-normalised MFCC **distance** |
| `shift.mss` | Multi-scale spectrogram **distance** |

Everything below is the real production path: the real writer, the real renderer, the real
`synth-setter-add-embeddings` CLI. Nothing is faked or mocked.

In [ ]:
import subprocess
import sys
import tempfile
from pathlib import Path

import lance
import numpy as np

from synth_setter.data.vst.shapes import AUDIO_FIELD, SHIFT_FIELD
from synth_setter.data.vst.writers import make_lance_dataset
from synth_setter.param_spec_name import ParamSpecName
from synth_setter.pipeline.schemas.spec import RenderConfig
from synth_setter.synth_spec import SynthName, SynthSpec

ROWS = 50
SYNTH = "torchsynth_adsr"
SAMPLE_RATE = 22_050
DURATION_SECONDS = 0.5
CHANNELS = 2
VELOCITY = 100
SHIFT_SEED = 20260730

## 1. Render a 50-row torchsynth dataset

torchsynth renders in-process, so this needs no plugin bundle and no R2 credentials.
`make_lance_dataset` is the same writer the distributed pipeline calls per shard.

In [ ]:
render_config = RenderConfig(
    synth=SynthSpec(
        name=SynthName(SYNTH),
        param_spec_name=ParamSpecName(SYNTH),
        plugin_path="torchsynth",
        plugin_state_path="",
        synth_version="1.0.2",
    ),
    renderer_backend="torchsynth",
    sample_rate=SAMPLE_RATE,
    channels=CHANNELS,
    velocity=VELOCITY,
    signal_duration_seconds=DURATION_SECONDS,
    min_loudness=-70.0,
    samples_per_render_batch=8,
    samples_per_shard=ROWS,
    base_seed=42,
    plugin_reload_cadence="once",
    gui_toggle_cadence="never",
)

workdir = Path(tempfile.mkdtemp(prefix="param-shift-walkthrough-"))
uri = workdir / "shard-000000.lance"
make_lance_dataset(uri, render_config)

dataset = lance.dataset(str(uri))
print(f"{dataset.count_rows()} rows -> {uri}")
print("columns:", dataset.schema.names)

## 2. Add the `shift` column

The CLI runs as a real subprocess, exactly as an operator would invoke it. `render=` and
`synth=` compose the renderer the re-render goes through (#2565); the param spec comes from
that synth identity, so a shift can never be encoded against a spec the renderer does not
share.

`param_shift_seed` is namespaced away from the dataset's `base_seed`, so reusing the same
number here is harmless — the replacement draws come from a stream datagen never touches.

In [ ]:
subprocess.run(  # noqa: S603 — sys.executable and every argument are notebook-owned
    [
        sys.executable,
        "-m",
        "synth_setter.pipeline.data.add_embeddings",
        f"lance_uri={uri}",
        "embeddings=[param_shift]",
        "render=torchsynth",
        f"synth={SYNTH}",
        f"render.sample_rate={SAMPLE_RATE}",
        f"render.channels={CHANNELS}",
        f"render.velocity={VELOCITY}",
        f"render.signal_duration_seconds={DURATION_SECONDS}",
        f"param_shift_seed={SHIFT_SEED}",
        "batch_size=10",
        "build_index=false",
    ],
    check=True,
)

dataset = lance.dataset(str(uri))
print(dataset.schema.field(SHIFT_FIELD))

## 3. Read the shift back

Lance projects struct subfields directly, so a query for `shift.param` and `shift.mss`
never materialises the re-rendered audio.

In [ ]:
scores = dataset.to_table(
    columns={
        "param": f"{SHIFT_FIELD}.param",
        "amount": f"{SHIFT_FIELD}.amount",
        "rms": f"{SHIFT_FIELD}.rms",
        "sot": f"{SHIFT_FIELD}.sot",
        "wmfcc": f"{SHIFT_FIELD}.wmfcc",
        "mss": f"{SHIFT_FIELD}.mss",
    }
).to_pandas()

scores.head(10)

### Rows are spread evenly across the spec

Assignment keys on the Lance row id, so each parameter owns the same share of rows give or
take one — and the same row always draws the same replacement, including after a
resume-cache replay.

In [ ]:
scores["param"].value_counts().sort_index()

### Which parameters move the sound most

This is the question the column exists to answer: per parameter, how far does the audio
travel when only that knob is redrawn. `rms` is a similarity (lower = more changed); the
other three are distances (higher = more changed).

In [ ]:
summary = (
    scores.groupby("param")[["amount", "rms", "sot", "wmfcc", "mss"]]
    .mean()
    .sort_values("mss", ascending=False)
)
summary

A zero `amount` is legitimate rather than a bug: `pitch` is discrete, so a redraw can land
back on its original value, and such a row should score as unchanged — `rms` at 1.0 and the
three distances at 0. At 50 rows this is usually empty (`pitch` owns about a seventh of the
rows and has 25 values, so roughly 0.3 rows are expected); the check is here because a
*non*-empty result whose scores were not ~unchanged would mean the recorded `amount` and the
recorded audio disagree.

In [ ]:
unchanged = scores[scores["amount"] == 0.0]
print(f"{len(unchanged)} row(s) redrew their original value: {unchanged['param'].tolist()}")
unchanged

### The recorded audio really is the recorded patch

`shift.audio` is stored exactly like `audio` — same shape, same dtype — so the pair is
directly comparable.

In [ ]:
audio = dataset.to_table(columns=[AUDIO_FIELD]).column(AUDIO_FIELD)
audio = audio.combine_chunks().to_numpy_ndarray()
shift_audio = (
    dataset.to_table(columns={"a": f"{SHIFT_FIELD}.audio"})
    .column("a")
    .combine_chunks()
    .to_numpy_ndarray()
)

print("audio      ", audio.shape, audio.dtype)
print("shift.audio", shift_audio.shape, shift_audio.dtype)

loudest = int(np.argmax(scores["mss"].to_numpy()))
print(
    f"\nrow {loudest} shifted {scores['param'].iloc[loudest]!r} "
    f"by {scores['amount'].iloc[loudest]:.4f} -> mss {scores['mss'].iloc[loudest]:.3f}"
)

Listen to the biggest mover — the original patch, then the same patch with one parameter
redrawn.

In [ ]:
from IPython.display import Audio, display

display(Audio(audio[loudest].astype(np.float32), rate=SAMPLE_RATE))
display(Audio(shift_audio[loudest].astype(np.float32), rate=SAMPLE_RATE))

## 4. Browse it in SmooSense

[SmooSense](https://smoosense.ai) renders the Lance table as an interactive grid in the
notebook — per-column histograms, filters, and sorting. Point it at the `.lance` dataset
directly rather than converting through pandas, so the native column types survive.

Sort by `shift.mss` to rank rows by how much the redraw moved the sound, or group by
`shift.param` to compare parameters against each other.

> SmooSense is an optional in-notebook viewer, not a pipeline dependency; it is installed by
> the project `notebooks` dependency group with the `jupyter` extra
> ([#1681](https://github.com/tinaudio/synth-setter/issues/1681)).

In [ ]:
from IPython.display import IFrame
from smoosense.widget import _SmooSenseServer

server = _SmooSenseServer()
server.start_if_needed()
IFrame(f"{server.base_url}/Table?tablePath={uri}", width="100%", height=900)